# Task 04 — Static graph contracts and ONNX

**Purpose.** Export a prefill graph and a decode graph from a tiny Qwen3,
read their frozen input/output contracts, and prove the ONNX graphs
compute the same numbers as PyTorch.

**Lesson.** Open `../docs/tasks/04-static-graph-contracts-and-onnx.html`
and read it first.

**Environment.** Use this repo's `.venv` kernel. The tiny-model parts run
offline. The last part reads the cached Qwen3-0.6B config from
`/Volumes/T9`, so the drive must be mounted. ONNX files go to
`artifacts/onnx/` (gitignored). Small evidence goes to `results/`.


In [1]:
# Setup. Set HF_HOME BEFORE importing transformers, or the cache moves.
import os, sys, json
from pathlib import Path

assert os.path.ismount("/Volumes/T9"), "Mount the T9 drive first (df -h /Volumes/T9)"
assert "transformers" not in sys.modules, "Restart the kernel: transformers was imported before HF_HOME was set"
os.environ["HF_HOME"] = "/Volumes/T9/qualcomm-edge-slm-lab/hf-home"

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = REPO / "results"
ONNX_DIR = REPO / "artifacts" / "onnx"
ONNX_DIR.mkdir(parents=True, exist_ok=True)

try:
    from edge_slm_lab.static_wrapper import (
        tiny_qwen3, export_prefill, export_decode,
        PrefillWrapper, DecodeWrapper, past_names,
    )
    from edge_slm_lab.onnx_tools import graph_io, op_counts, all_shapes_fixed, contract
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "edge_slm_lab is not installed. In a terminal run: uv pip install -e .  "
        "Then restart this kernel."
    ) from e

import numpy as np
import torch
import onnxruntime as ort

print("setup ok — ONNX files ->", ONNX_DIR)

/Users/chayut/projects/qualcomm-edge-slm-lab/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup ok — ONNX files -> /Users/chayut/projects/qualcomm-edge-slm-lab/artifacts/onnx


## Part 1 — Build the tiny model and export two graphs

Same layer type and cache layout as Qwen3-0.6B, just smaller numbers.
Small means each export takes seconds and the files stay tiny.


In [2]:
model = tiny_qwen3()
cfg = model.config
print(f"layers={cfg.num_hidden_layers}  query_heads={cfg.num_attention_heads}  "
      f"kv_heads={cfg.num_key_value_heads}  head_dim={cfg.head_dim}  vocab={cfg.vocab_size}")

layers=2  query_heads=4  kv_heads=2  head_dim=16  vocab=128


In [10]:
# PARAMETER — run everything below with 8 first.
# Then set 16 and rerun from THIS cell down through Part 4.
PROMPT_LEN = 16

In [11]:
# Export. The TracerWarnings are expected: the trace is frozen at one
# shape on purpose. That frozen shape IS the contract.
prefill_path = ONNX_DIR / f"tiny_prefill_p{PROMPT_LEN}.onnx"
decode_path = ONNX_DIR / f"tiny_decode_past{PROMPT_LEN}.onnx"
export_prefill(model, prefill_path, prompt_len=PROMPT_LEN)
export_decode(model, decode_path, past_len=PROMPT_LEN)
for p in (prefill_path, decode_path):
    print(f"{p.name}: {p.stat().st_size / 1024:.0f} KiB")

tiny_prefill_p16.onnx: 415 KiB
tiny_decode_past16.onnx: 415 KiB


## Part 2 — Read the declared contract

`graph_io` reads only the ONNX header: every input and output with its
name, dtype, and shape. Look at the shapes. Every dimension is a plain
integer. Nothing is dynamic.


In [12]:
def show(path, title):
    io = graph_io(path)
    print(f"--- {title} ---")
    for kind in ("inputs", "outputs"):
        for t in io[kind]:
            print(f"{kind[:-1]:>7}  {t['name']:<16} {t['dtype']:<8} {t['shape']}")
    return io

pre_io = show(prefill_path, f"prefill graph (P={PROMPT_LEN})")
dec_io = show(decode_path, f"decode graph (past={PROMPT_LEN})")

assert all_shapes_fixed(pre_io) and all_shapes_fixed(dec_io)
pre_shapes = {t["name"]: t["shape"] for t in pre_io["inputs"] + pre_io["outputs"]}
dec_shapes = {t["name"]: t["shape"] for t in dec_io["inputs"] + dec_io["outputs"]}
assert pre_shapes["input_ids"] == [1, PROMPT_LEN]
assert pre_shapes["present_key_0"] == [1, 2, PROMPT_LEN, 16]
assert dec_shapes["input_ids"] == [1, 1]
assert dec_shapes["past_key_0"] == [1, 2, PROMPT_LEN, 16]
assert dec_shapes["present_key_0"] == [1, 2, PROMPT_LEN + 1, 16]
print("contract asserts ok: decode cache goes in at", PROMPT_LEN, "and comes out at", PROMPT_LEN + 1)

# Save evidence now, keyed by prompt length, so a rerun with 16 merges in.
contracts_file = RESULTS / "04_contracts.json"
data = json.loads(contracts_file.read_text()) if contracts_file.exists() else {}
data[str(PROMPT_LEN)] = {
    "prefill": contract(prefill_path, f"prefill_p{PROMPT_LEN}"),
    "decode": contract(decode_path, f"decode_past{PROMPT_LEN}"),
}
contracts_file.write_text(json.dumps(data, indent=2))
print("saved ->", contracts_file)

--- prefill graph (P=16) ---
  input  input_ids        int64    [1, 16]
 output  logits           float    [1, 16, 128]
 output  present_key_0    float    [1, 2, 16, 16]
 output  present_value_0  float    [1, 2, 16, 16]
 output  present_key_1    float    [1, 2, 16, 16]
 output  present_value_1  float    [1, 2, 16, 16]
--- decode graph (past=16) ---
  input  input_ids        int64    [1, 1]
  input  past_key_0       float    [1, 2, 16, 16]
  input  past_value_0     float    [1, 2, 16, 16]
  input  past_key_1       float    [1, 2, 16, 16]
  input  past_value_1     float    [1, 2, 16, 16]
 output  logits           float    [1, 1, 128]
 output  present_key_0    float    [1, 2, 17, 16]
 output  present_value_0  float    [1, 2, 17, 16]
 output  present_key_1    float    [1, 2, 17, 16]
 output  present_value_1  float    [1, 2, 17, 16]
contract asserts ok: decode cache goes in at 16 and comes out at 17
saved -> /Users/chayut/projects/qualcomm-edge-slm-lab/results/04_contracts.json


In [13]:
# What is inside the graph? Counts of node types. MatMul carries the
# projections and attention. No Python, no branches, no cache object.
for name, count in list(op_counts(decode_path).items())[:10]:
    print(f"{name:<12} {count}")

Constant     122
Mul          43
Cast         22
MatMul       20
Add          19
Reshape      12
Transpose    11
Concat       10
Slice        10
Where        9


## Part 3 — Same numbers as PyTorch?

An export is only useful if the graph computes what the model computes.
Tolerance for fp32: max abs difference < 1e-4. Typical value is ~1e-6.


In [14]:
TOLERANCE = 1e-4
torch.manual_seed(1)
prompt = torch.randint(0, cfg.vocab_size, (1, PROMPT_LEN))

with torch.no_grad():
    torch_pre = PrefillWrapper(model)(prompt)
sess = ort.InferenceSession(str(prefill_path))
onnx_pre = sess.run(None, {"input_ids": prompt.numpy()})

prefill_diff = max(float(np.abs(g - w.numpy()).max()) for g, w in zip(onnx_pre, torch_pre))
print(f"prefill: max abs diff over logits + all KV outputs = {prefill_diff:.2e}")
assert prefill_diff < TOLERANCE

prefill: max abs diff over logits + all KV outputs = 1.07e-06


In [15]:
# One decode step through ONNX Runtime, fed by the prefill outputs.
step = torch.tensor([[int(onnx_pre[0][0, -1].argmax())]])
past = [torch.from_numpy(t) for t in onnx_pre[1:]]

with torch.no_grad():
    torch_dec = DecodeWrapper(model)(step, *past)
dsess = ort.InferenceSession(str(decode_path))
feed = {"input_ids": step.numpy()}
for name, tensor in zip(past_names(cfg.num_hidden_layers), onnx_pre[1:]):
    feed[name] = tensor
onnx_dec = dsess.run(None, feed)

decode_diff = max(float(np.abs(g - w.numpy()).max()) for g, w in zip(onnx_dec, torch_dec))
torch_next = int(torch_dec[0][0, -1].argmax())
onnx_next = int(onnx_dec[0][0, -1].argmax())
print(f"decode: max abs diff = {decode_diff:.2e}")
print(f"next token id: torch={torch_next}  onnx={onnx_next}")
assert decode_diff < TOLERANCE and torch_next == onnx_next

match_file = RESULTS / "04_numeric_match.json"
data = json.loads(match_file.read_text()) if match_file.exists() else {}
data[str(PROMPT_LEN)] = {
    "tolerance": TOLERANCE,
    "prefill_max_abs_diff": prefill_diff,
    "decode_max_abs_diff": decode_diff,
    "next_token_id_matches": torch_next == onnx_next,
}
match_file.write_text(json.dumps(data, indent=2))
print("saved ->", match_file)

decode: max abs diff = 5.96e-07
next token id: torch=100  onnx=100
saved -> /Users/chayut/projects/qualcomm-edge-slm-lab/results/04_numeric_match.json


## Part 4 — Break the contract on purpose

Feed a prompt one token longer than the graph was frozen at. The runtime
must refuse it. Read the error: it names the tensor, the index, and both
sizes. A compiled NPU bundle enforces its shapes the same way.


In [16]:
longer = np.zeros((1, PROMPT_LEN + 1), dtype=np.int64)
try:
    sess.run(None, {"input_ids": longer})
    raise AssertionError("the graph accepted a wrong-length input — it should not")
except Exception as e:
    error_text = str(e)
    print(error_text[:300])
assert "Got invalid dimensions" in error_text

data = json.loads(match_file.read_text())
data[str(PROMPT_LEN)]["wrong_shape_rejected"] = True
match_file.write_text(json.dumps(data, indent=2))
print("\nsaved: wrong_shape_rejected = True")

[ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Got invalid dimensions for input: input_ids for the following indices
 index: 1 Got: 17 Expected: 16
 Please fix either the inputs/outputs or the model.

saved: wrong_shape_rejected = True


## Now change the parameter

Go back to the `PROMPT_LEN = 8` cell, set it to **16**, and rerun from
there through Part 4. Watch two things:

- every frozen shape in Part 2 changes — a new length means a new graph;
- the graph exported at 8 still exists and still only accepts 8.

That is why one compiled bundle ships several fixed graphs and a fixed
context limit. Then continue below.


## Part 5 — The real Qwen3-0.6B contract, computed from its config

No export here. Full-model export is heavy and Qualcomm's adapter does it
properly (next task). The config alone tells us the full interface.


In [17]:
from transformers import AutoConfig

qcfg = AutoConfig.from_pretrained("Qwen/Qwen3-0.6B")
assert (qcfg.num_hidden_layers, qcfg.num_key_value_heads, qcfg.head_dim) == (28, 8, 128)

P = 24  # the Task 02/03 chat prompt length
kv = [1, qcfg.num_key_value_heads, P, qcfg.head_dim]
kv_out = [1, qcfg.num_key_value_heads, P + 1, qcfg.head_dim]
real = {
    "model": "Qwen/Qwen3-0.6B",
    "example_prompt_len": P,
    "prefill": {
        "num_inputs": 1,
        "num_outputs": 1 + 2 * qcfg.num_hidden_layers,
        "input_ids": [1, P],
        "logits": [1, P, qcfg.vocab_size],
        "present_key_i / present_value_i (x28 layers)": kv,
    },
    "decode": {
        "num_inputs": 1 + 2 * qcfg.num_hidden_layers,
        "num_outputs": 1 + 2 * qcfg.num_hidden_layers,
        "input_ids": [1, 1],
        "past_key_i / past_value_i (x28 layers)": kv,
        "logits": [1, 1, qcfg.vocab_size],
        "present_key_i / present_value_i (x28 layers)": kv_out,
    },
}
print(json.dumps(real, indent=2))
print(f"\ndecode graph interface: {real['decode']['num_inputs']} inputs, "
      f"{real['decode']['num_outputs']} outputs — the cache IS the interface")

data = json.loads(contracts_file.read_text())
data["qwen3_0_6b_from_config"] = real
contracts_file.write_text(json.dumps(data, indent=2))
print("saved ->", contracts_file)

{
  "model": "Qwen/Qwen3-0.6B",
  "example_prompt_len": 24,
  "prefill": {
    "num_inputs": 1,
    "num_outputs": 57,
    "input_ids": [
      1,
      24
    ],
    "logits": [
      1,
      24,
      151936
    ],
    "present_key_i / present_value_i (x28 layers)": [
      1,
      8,
      24,
      128
    ]
  },
  "decode": {
    "num_inputs": 57,
    "num_outputs": 57,
    "input_ids": [
      1,
      1
    ],
    "past_key_i / past_value_i (x28 layers)": [
      1,
      8,
      24,
      128
    ],
    "logits": [
      1,
      1,
      151936
    ],
    "present_key_i / present_value_i (x28 layers)": [
      1,
      8,
      25,
      128
    ]
  }
}

decode graph interface: 57 inputs, 57 outputs — the cache IS the interface
saved -> /Users/chayut/projects/qualcomm-edge-slm-lab/results/04_contracts.json


## Final summary

Reads the evidence files back from disk, so it works even after a kernel
restart. All checks must be True. Also run the tests in a terminal:

```bash
uv run pytest tests/test_shape_contracts.py -q
```


In [18]:
contracts = json.loads((RESULTS / "04_contracts.json").read_text())
matches = json.loads((RESULTS / "04_numeric_match.json").read_text())
len_keys = {k for k in contracts if k != "qwen3_0_6b_from_config"}

CHECKS = {
    "module_import_ok": True,
    "contracts_all_fixed": all(
        contracts[k][g]["all_shapes_fixed"] for k in len_keys for g in ("prefill", "decode")
    ),
    "prefill_numeric_match": all(
        m["prefill_max_abs_diff"] < m["tolerance"] for m in matches.values()
    ),
    "decode_numeric_match": all(
        m["decode_max_abs_diff"] < m["tolerance"] and m["next_token_id_matches"]
        for m in matches.values()
    ),
    "wrong_shape_rejected": all(m.get("wrong_shape_rejected") for m in matches.values()),
    "two_prompt_lens_saved": {"8", "16"} <= len_keys,
    "real_config_contract_saved": "qwen3_0_6b_from_config" in contracts,
}
for name, ok in CHECKS.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

summary = {
    "task": 4,
    "checks": CHECKS,
    "artifacts": ["results/04_contracts.json", "results/04_numeric_match.json"],
}
(RESULTS / "04_summary.json").write_text(json.dumps(summary, indent=2))
print("\nsaved -> results/04_summary.json")
assert all(CHECKS.values()), "some checks failed — see FAIL lines above"
print("all checks passed")

PASS  module_import_ok
PASS  contracts_all_fixed
PASS  prefill_numeric_match
PASS  decode_numeric_match
PASS  wrong_shape_rejected
PASS  two_prompt_lens_saved
PASS  real_config_contract_saved

saved -> results/04_summary.json
all checks passed


## Part 6 (optional) — One graph, many steps

Parts 1–4 froze the cache length into the graph, so each decode step
needed its own graph. Real bundles use ONE decode graph. The trick:
move everything that varies from **shapes** into **values**.

- The cache input is always full size: `[1, 2, MAX_LEN, 16]`, empty
  slots zero-padded.
- An `attention_mask` input hides the empty slots (large negative
  number added to attention scores before softmax).
- A `position_ids` input tells RoPE where the new token sits.
- The graph outputs only the new token's K/V (`new_key_i`,
  `new_value_i`, shape `[1, 2, 1, 16]`). The HOST writes it into the
  next free slot — a real runtime does this same bookkeeping.

This part is outside the completion gate. Run it if you want to see the
single-graph loop reproduce Task 03's dynamic loop token for token.


In [19]:
# Export ONE padded decode graph. MAX_LEN is the compile-time context limit.
from edge_slm_lab.static_wrapper import export_padded_decode, padded_mask, PaddedDecodeWrapper
from edge_slm_lab.manual_decode import greedy_decode

MAX_LEN = 32
padded_path = ONNX_DIR / f"tiny_decode_padded_max{MAX_LEN}.onnx"
export_padded_decode(model, padded_path, max_len=MAX_LEN)

pio = graph_io(padded_path)
assert all_shapes_fixed(pio)
for kind in ("inputs", "outputs"):
    for t in pio[kind][:5]:
        print(f"{kind[:-1]:>7}  {t['name']:<16} {t['dtype']:<8} {t['shape']}")
    if len(pio[kind]) > 5:
        print(f"        ... {len(pio[kind]) - 5} more {kind}")
print("\nEvery shape is still fixed. The cache is always full size;")
print("the mask and position INPUT VALUES change per step, not the shapes.")

  input  input_ids        int64    [1, 1]
  input  position_ids     int64    [1, 1]
  input  attention_mask   float    [1, 1, 1, 33]
  input  past_key_0       float    [1, 2, 32, 16]
  input  past_value_0     float    [1, 2, 32, 16]
        ... 2 more inputs
 output  logits           float    [1, 1, 128]
 output  new_key_0        float    [1, 2, 1, 16]
 output  new_value_0      float    [1, 2, 1, 16]
 output  new_key_1        float    [1, 2, 1, 16]
 output  new_value_1      float    [1, 2, 1, 16]

Every shape is still fixed. The cache is always full size;
the mask and position INPUT VALUES change per step, not the shapes.


In [20]:
# The loop: prefill once, then N_STEPS tokens through the SAME session.
# Reference: the Task 03 dynamic loop on the same prompt.
N_STEPS = 12
torch.manual_seed(3)
loop_prompt = torch.randint(0, cfg.vocab_size, (1, PROMPT_LEN))
assert PROMPT_LEN + N_STEPS <= MAX_LEN, "would overflow the compile-time context limit"

ref_ids, _ = greedy_decode(model, loop_prompt, max_new_tokens=N_STEPS)

with torch.no_grad():
    pre = PrefillWrapper(model)(loop_prompt)

# Host-side padded buffers: real K/V in slots 0..PROMPT_LEN-1, zeros after.
past_bufs = [np.zeros((1, cfg.num_key_value_heads, MAX_LEN, cfg.head_dim), dtype=np.float32)
             for _ in range(2 * cfg.num_hidden_layers)]
for buf, t in zip(past_bufs, pre[1:]):
    buf[:, :, :PROMPT_LEN] = t.numpy()

psess = ort.InferenceSession(str(padded_path))
filled = PROMPT_LEN
next_id = int(pre[0][0, -1].argmax())
onnx_ids = [next_id]
for _ in range(N_STEPS - 1):
    feed = {
        "input_ids": np.array([[next_id]], dtype=np.int64),
        "position_ids": np.array([[filled]], dtype=np.int64),
        "attention_mask": padded_mask(filled, MAX_LEN).numpy(),
    }
    for name, buf in zip(past_names(cfg.num_hidden_layers), past_bufs):
        feed[name] = buf
    outs = psess.run(None, feed)
    for buf, new in zip(past_bufs, outs[1:]):
        buf[:, :, filled] = new[:, :, 0]  # host writes the next free slot
    filled += 1
    next_id = int(outs[0][0, -1].argmax())
    onnx_ids.append(next_id)

print("dynamic loop (Task 03):", ref_ids)
print("one static graph:      ", onnx_ids)
assert onnx_ids == ref_ids
print(f"\nmatch — {N_STEPS} tokens through one frozen graph, "
      f"cache slots used: {filled} of {MAX_LEN}")

(RESULTS / "04_padded_loop.json").write_text(json.dumps({
    "optional": True,
    "max_len": MAX_LEN,
    "prompt_len": PROMPT_LEN,
    "n_steps": N_STEPS,
    "ids_match_dynamic_loop": onnx_ids == ref_ids,
    "cache_slots_used": filled,
}, indent=2))
print("saved -> results/04_padded_loop.json")

dynamic loop (Task 03): [38, 100, 96, 108, 12, 15, 21, 15, 21, 15, 21, 15]
one static graph:       [38, 100, 96, 108, 12, 15, 21, 15, 21, 15, 21, 15]

match — 12 tokens through one frozen graph, cache slots used: 27 of 32
saved -> results/04_padded_loop.json


**What to notice.** The session was created once and never re-exported.
Step 5 and step 12 ran the identical graph — only `position_ids`, the
mask values, and the buffer contents differed. And the limit is real:
the assert above refuses `PROMPT_LEN + N_STEPS > MAX_LEN`, because slot
`MAX_LEN` does not exist. That is a context limit, chosen at export
time. Try `N_STEPS = 40` to see it refuse.
